# Sentinel-2 Mosaikierung – Bono-Region, Januar 2025

Dieses Notebook verarbeitet die 6 von Google Earth Engine exportierten Sentinel-2-Kacheln der Bono/Bono-East-Region (Ghana) zu einem einzigen Gesamtbild (Mosaic).

**Bänder (in dieser Reihenfolge im TIF):**
| Index | Band | Wellenlänge | Beschreibung |
|-------|------|-------------|-------------|
| 0 | B2 | ~492 nm | Blau |
| 1 | B3 | ~560 nm | Grün |
| 2 | B4 | ~665 nm | Rot |
| 3 | B8A | ~865 nm | Schmales NIR |
| 4 | B11 | ~1610 nm | SWIR 1 |
| 5 | B12 | ~2190 nm | SWIR 2 |

> **Hinweis:** Die Werte wurden bereits in GEE durch 10.000 geteilt und liegen als Float im Bereich 0–1 vor.

In [1]:
# ==============================================================================
# 1. IMPORTE
# ==============================================================================
import rasterio
from rasterio.merge import merge
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# ==============================================================================
# 2. DATEIPFADE DER 6 KACHELN
# ==============================================================================
file_paths = [
    "/Users/mathias/dev/BA_Thesis_SmallMinesDS/data/raw/Sentinel2_Bono_Januar2025_10m-0000000000-0000000000.tif",
    "/Users/mathias/dev/BA_Thesis_SmallMinesDS/data/raw/Sentinel2_Bono_Januar2025_10m-0000000000-0000013568.tif",
    "/Users/mathias/dev/BA_Thesis_SmallMinesDS/data/raw/Sentinel2_Bono_Januar2025_10m-0000000000-0000027136.tif",
    "/Users/mathias/dev/BA_Thesis_SmallMinesDS/data/raw/Sentinel2_Bono_Januar2025_10m-0000013568-0000000000.tif",
    "/Users/mathias/dev/BA_Thesis_SmallMinesDS/data/raw/Sentinel2_Bono_Januar2025_10m-0000013568-0000013568.tif",
    "/Users/mathias/dev/BA_Thesis_SmallMinesDS/data/raw/Sentinel2_Bono_Januar2025_10m-0000013568-0000027136.tif"
]

print("Öffne die 6 Kacheln...")

src_files_to_mosaic = []
for fp in file_paths:
    src = rasterio.open(fp)
    src_files_to_mosaic.append(src)

print(f"Anzahl Kacheln: {len(src_files_to_mosaic)}")
print(f"Bänder pro Kachel: {src_files_to_mosaic[0].count}")
print(f"Datentyp: {src_files_to_mosaic[0].dtypes[0]}")
print(f"CRS: {src_files_to_mosaic[0].crs}")

Öffne die 6 Kacheln...
Anzahl Kacheln: 6
Bänder pro Kachel: 6
Datentyp: float32
CRS: EPSG:32630


In [3]:
# ==============================================================================
# 3. DAS MOSAIK ERSTELLEN (Merging) – nur wenn noch nicht vorhanden
# ==============================================================================
import os

output_path = "/Users/mathias/dev/BA_Thesis_SmallMinesDS/data/raw/Bono_Merged_2025.tif"

if os.path.exists(output_path):
    print(f"Merged-Datei bereits vorhanden ({os.path.getsize(output_path) / 1e9:.1f} GB), überspringe Merge.")
    for src in src_files_to_mosaic:
        src.close()
else:
    print("Klebe Bilder zusammen (kann mehrere Minuten dauern, ~24 GB RAM nötig)...")
    mosaic, out_trans = merge(src_files_to_mosaic)
    print(f"Mosaic-Shape: {mosaic.shape}  (Bänder x Höhe x Breite)")

    out_meta = src_files_to_mosaic[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans
    })

    print(f"Speichere als: {output_path}")
    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(mosaic)

    for src in src_files_to_mosaic:
        src.close()
    del mosaic
    print("Gespeichert. RAM freigegeben.")

Klebe Bilder zusammen (das kann ein paar Sekunden dauern)...
Mosaic-Shape: (6, 26886, 36793)  (Bänder x Höhe x Breite)
Wertebereich: min=nan, max=nan


In [ ]:
# ==============================================================================
# 4. DATEI-INFOS PRÜFEN (kein RAM-Aufwand)
# ==============================================================================
output_path = "/Users/mathias/dev/BA_Thesis_SmallMinesDS/data/raw/Bono_Merged_2025.tif"

with rasterio.open(output_path) as src:
    print(f"Datei:    {output_path}")
    print(f"Größe:    {src.width} x {src.height} Pixel")
    print(f"Bänder:   {src.count}")
    print(f"Datentyp: {src.dtypes[0]}")
    print(f"CRS:      {src.crs}")
    print(f"Auflösung: {src.res[0]:.1f} m/Pixel")

Speichere das Gesamtbild als: /Users/mathias/dev/BA_Thesis_SmallMinesDS/data/raw/Bono_Merged_2025.tif
Gespeichert und Dateien geschlossen.


: 

## Visualisierung

**Echtfarben (RGB):** B4 (Rot) + B3 (Grün) + B2 (Blau) – so sieht das Bild für das menschliche Auge aus.

**Falschfarben (SWIR/NIR/Rot):** B11 + B8A + B4 – in dieser Darstellung erscheinen Galamsey-Minen (nackte Erde, Schlammpfützen) hell türkis/weißlich und heben sich stark vom dunklen Wald ab. Entspricht dem GEE-Vorschau-Layer im Export-Script.

In [ ]:
# ==============================================================================
# 5. VISUALISIERUNG – RAM-schonend durch Downsampling beim Lesen
# ==============================================================================
# Das Originalbild ist ~27.000 x 37.000 Pixel = ~1 Mrd. Pixel → Kernel-Crash.
# Wir lesen es direkt auf 2% der Originalgröße heruntergerechnet (rasterio macht
# das intern, ohne das volle Bild in den RAM zu laden).

from rasterio.enums import Resampling

SCALE = 0.04  # 4% der Originalgröße → ~1.075 x 1.472 Pixel, ~50 MB RAM

output_path = "/Users/mathias/dev/BA_Thesis_SmallMinesDS/data/raw/Bono_Merged_2025.tif"

print("Lese herunterskaliertes Vorschaubild...")
with rasterio.open(output_path) as src:
    h = int(src.height * SCALE)
    w = int(src.width  * SCALE)
    data = src.read(
        out_shape=(src.count, h, w),
        resampling=Resampling.average
    )

print(f"Geladene Vorschau-Shape: {data.shape}  ({data.nbytes / 1e6:.0f} MB im RAM)")

blue  = data[0]
green = data[1]
red   = data[2]
nir   = data[3]
swir1 = data[4]

rgb_bright         = np.clip(np.dstack((red, green, blue)) * 3.5, 0, 1)
false_color_bright = np.clip(np.dstack((swir1, nir, red)) * 3.5, 0, 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 12))

ax1.imshow(rgb_bright)
ax1.set_title("Gesamte Bono-Region: Echtfarben (B4/B3/B2)", fontsize=14)
ax1.axis('off')

ax2.imshow(false_color_bright)
ax2.set_title("Gesamte Bono-Region: Falschfarben (B11/B8A/B4)", fontsize=14)
ax2.axis('off')

plt.tight_layout()
plt.show()
print("Fertig!")

Erstelle die Visualisierung der kompletten Region...
